> 在线讲义：[第 3 章 · 先算下限：读权重要多久、算一遍要多久、放不放得下](https://wikieden.github.io/robot-edge-inference/lessons/03-lower-bounds.html) · 本书《AI Infra 扩展 · 机器人端侧推理》

# 第 3 章 · 先算下限：读权重要多久、算一遍要多久、放不放得下

配套讲义：仓库里 `ai-infra-edge/notes/lessons/03-lower-bounds.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import math
import random

print("纯 Python，不需要安装任何库")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（OpenVLA、π0 的数来自论文；48 个文字 token、利用率 40%、预留 3 GB 是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 4090 一步下限 | $7 \times 15 \div 1008$ | `openvla_step_ms(15, 1008)` | 104.2 ms → 9.6 Hz |
| 第二步 AGX Orin bf16 | $7 \times 15 \div 204.8$ | `openvla_step_ms(15, 204.8)` | 512.7 ms → 1.95 Hz |
| 第二步 AGX Orin int4 | $7 \times 7.0 \div 204.8$ | `openvla_step_ms(7.0, 204.8)` | 239.3 ms → 4.18 Hz |
| 第三步 前缀运算量 | $2 \times 3\text{e}9 \times 816$ | `prefill_flops(3e9, 816)` | 4.90 万亿次 |
| 第三步 反推 4090 有效算力 | 运算量 ÷ 46 ms | `flops / 0.046` | 106 万亿次 / 秒 |
| 第三步 AGX Orin 前缀 | 运算量 ÷ 55 万亿 | `compute_ms(flops, 55.2)` | 89 ms |
| 第四步 放不放得下 | 占用 ≤ 总 − 3 | `fits(7.0, 16)` | 放得下 |

OpenVLA 15 GB / 7.0 GB、6 Hz，π0 的 46 ms 来自论文；带宽来自官方页面和 AII 的硬件表。
π0 前缀里"48 个文字 token"、利用率 40% 是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
BW = {"RTX 4090": 1008, "AGX Orin 64GB": 204.8, "Jetson T5000（Thor）": 273, "Orin NX 16GB": 102.4}  # GB/s

**函数 `read_ms`**：读一遍 gb 字节要多久：t = 字节 ÷ 带宽。

In [ ]:
def read_ms(gb: float, bw_gbps: float) -> float:
    """读一遍 gb 字节要多久：t = 字节 ÷ 带宽。"""
    return gb / bw_gbps * 1000

**函数 `compute_ms`**：算一遍要多久：t = 运算量 ÷ 算力（tflops 是每秒万亿次）。

In [ ]:
def compute_ms(flops: float, tflops: float) -> float:
    """算一遍要多久：t = 运算量 ÷ 算力（tflops 是每秒万亿次）。"""
    return flops / (tflops * 1e12) * 1000

**函数 `openvla_step_ms`**：OpenVLA 一步要逐个吐 7 个动作 token，每吐一个都要把权重读一遍。

In [ ]:
def openvla_step_ms(model_gb: float, bw_gbps: float, n_tokens: int = 7) -> float:
    """OpenVLA 一步要逐个吐 7 个动作 token，每吐一个都要把权重读一遍。"""
    return n_tokens * read_ms(model_gb, bw_gbps)

**函数 `prefill_flops`**：前缀（图像 + 文字）一次算完：运算量 ≈ 2 × 参数量 × token 数。

In [ ]:
def prefill_flops(params: float, tokens: int) -> float:
    """前缀（图像 + 文字）一次算完：运算量 ≈ 2 × 参数量 × token 数。"""
    return 2 * params * tokens

**函数 `fits`**：放不放得下：模型 ≤ 总内存 − 系统预留。

In [ ]:
def fits(model_gb: float, mem_gb: float, reserve_gb: float = 3.0) -> bool:
    """放不放得下：模型 ≤ 总内存 − 系统预留。"""
    return model_gb <= mem_gb - reserve_gb

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步：OpenVLA（7B，bf16 占 15 GB）一步的带宽下限")
t = openvla_step_ms(15, BW["RTX 4090"])
print(f"  RTX 4090：7 × 15 GB ÷ 1008 GB/s = {t:.1f} ms → 最多 {1000 / t:.1f} Hz（论文实测约 6 Hz）")
for dev in ("AGX Orin 64GB", "Jetson T5000（Thor）"):
    for label, gb in (("bf16 15 GB", 15), ("int4 7.0 GB", 7.0)):
        t = openvla_step_ms(gb, BW[dev])
        print(f"  {dev:<20} {label:<12} {t:6.1f} ms → 最多 {1000 / t:.2f} Hz")

In [ ]:
print("\n第三步：π0 的前缀是算力受限（3 张图 × 256 token + 48 个文字 token，48 是编的）")
tokens = 3 * 256 + 48
flops = prefill_flops(3e9, tokens)
print(f"  token 数 {tokens}，运算量 ≈ 2 × 3e9 × {tokens} = {flops / 1e12:.2f} 万亿次")
print(f"  4090 实测前缀 14 + 32 = 46 ms → 反推有效算力 {flops / 0.046 / 1e12:.0f} 万亿次 / 秒")
print(f"  同一份权重读一遍只要 {read_ms(6.0, BW['RTX 4090']):.1f} ms（3B × 2 字节 = 6 GB）→ 前缀被算力卡住")
eff = 138 * 0.4
print(f"  AGX Orin：稠密 138 INT8 TOPS × 40%（编的）= {eff:.0f} → 前缀约 {compute_ms(flops, eff):.0f} ms")

In [ ]:
print("\n第四步：放不放得下（系统预留 3 GB 是编的）")
for dev, mem in (("Orin Nano 8GB", 8), ("Orin NX 16GB", 16), ("AGX Orin 64GB", 64)):
    print(f"  {dev:<14} bf16 15 GB：{'放得下' if fits(15, mem) else '放不下'}；"
          f"int4 7.0 GB：{'放得下' if fits(7.0, mem) else '放不下'}")

## 3. 代码里才看得到的

- `compute_ms` 的算力参数单位是"万亿次 / 秒"，函数里乘了 $10^{12}$；传 55 和传 55e12 结果差一万亿倍，单位最容易错。
- 带宽用的是 GB/s（$10^9$ 字节），模型大小也要用同一个 GB；如果模型大小写的是 GiB（$2^{30}$），要先换算，差约 7%。

## 4. 自己改一改

- 把 `BW` 里加一行 RTX 5090，查它的带宽填进去，看 OpenVLA 上限变多少。
- 把前缀 token 数改成 4 张图，看 AGX Orin 上前缀变成多少 ms。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 搬砖砌墙看更慢，读算两限取其大；
> 一步七个 token，权重读七遍；
> 前缀 token 多算力卡，逐个生成带宽卡；
> 先问放不放得下，再问跑得有多快。